In [1]:
from pathlib import Path
import pandas as pd

import teehr
from teehr import RemoteReadWriteEvaluation

from setup_utils import create_minio_spark_session

### Create spark session and init eval

In [2]:
spark = create_minio_spark_session()

INFO:teehr.evaluation.spark_session_utils:🚀 Creating Spark session: TEEHR Evaluation
INFO:teehr.evaluation.spark_session_utils:✅ Spark local configuration successful!
INFO:teehr.evaluation.spark_session_utils:Setting Hadoop's default AWS credentials provider and AWS region
INFO:teehr.evaluation.spark_session_utils:🔑 Using user-provided AWS credentials
INFO:teehr.evaluation.spark_session_utils:Configuring Iceberg catalogs...
INFO:teehr.evaluation.spark_session_utils:⚙️ All settings applied. Creating Spark session...
INFO:teehr.evaluation.spark_session_utils:🎉 Spark session created successfully!


In [3]:
ev = RemoteReadWriteEvaluation(spark=spark, enable_spark_proxy=True)

INFO:teehr.evaluation.evaluation:Using provided Spark session.
INFO:teehr.evaluation.evaluation:Active catalog set to iceberg.


### Load custom tables to warehouse

In [4]:
inputs_dir = Path(Path.cwd(), 'FIRO_data')

# load location_metrics from parquet
location_metrics_path = Path(inputs_dir, 'locations_metrics.parquet')
df = pd.read_parquet(location_metrics_path)
ev._load.dataframe(
    df=df,
    table_name='locations_metrics',
    write_mode='create_or_replace'
)

# load event_rankings from parquet
event_rankings_path = Path(inputs_dir, 'event_rankings.parquet')
df = pd.read_parquet(event_rankings_path)
ev._load.dataframe(
    df=df,
    table_name='event_rankings',
    write_mode='create_or_replace'
)

# load event_heatmap from parquet
event_heatmap_path = Path(inputs_dir, 'event_heatmap.parquet')
df = pd.read_parquet(event_heatmap_path)
ev._load.dataframe(
    df=df,
    table_name='event_heatmap',
    write_mode='create_or_replace'
)

INFO:teehr.evaluation.tables.generic_table:Getting table: locations_metrics.
INFO:teehr.evaluation.tables.base_table:Initializing Table for table: locations_metrics.
INFO:teehr.evaluation.tables.base_table:Loading files from iceberg.teehr.locations_metrics.
INFO:teehr.evaluation.read:Reading files from iceberg.teehr.locations_metrics.
INFO:teehr.evaluation.write:Start writing to warehouse table 'locations_metrics'.
INFO:teehr.evaluation.tables.generic_table:Getting table: locations_metrics.
INFO:teehr.evaluation.tables.base_table:Initializing Table for table: locations_metrics.
INFO:teehr.evaluation.tables.base_table:Loading files from iceberg.teehr.locations_metrics.
INFO:teehr.evaluation.read:Reading files from iceberg.teehr.locations_metrics.
INFO:teehr.evaluation.write:Finished writing to warehouse table 'locations_metrics' in 5.334 seconds.
INFO:teehr.evaluation.tables.generic_table:Getting table: event_rankings.
INFO:teehr.evaluation.tables.base_table:Initializing Table for table

In [5]:
# Set locations_metrics table properties for API queryables
table_name = 'locations_metrics'
group_by = [
    "primary_location_id",
    "configuration_name",
    "variable_name",
    "season",
    "forecast_lead_time",
    "forecast_lead_time_bin",
    "threshold",
]
metric_columns = [
    "mean_absolute_error",
    "root_mean_square_error",
    "relative_bias",
    "pearson_correlation",
    "nash_sutcliffe_efficiency",
    "FN", "TN", "FP", "TP",
    "probability_of_detection",
    "false_alarm_ratio",
    "critical_success_index",
    "frequency_bias_index",
    "mean_crps_ensemble",
    "mean_crps_ensemble_skill_score",
    "mean_brier_score",
    "mean_brier_score_skill_score",
]
properties = {
    "description": "FIRO forecast/hindcast performance metrics by location",
    "group_by": ", ".join(group_by),
    "metrics": ", ".join(metric_columns),
}
for key, value in properties.items():
    ev.spark.sql(f"""
        ALTER TABLE iceberg.teehr.{table_name} SET TBLPROPERTIES ('{key}' = '{value}')
    """)

In [6]:
# Set event_rankings table properties for API queryables
table_name = 'event_rankings'
group_by = [
    "primary_location_id",
    "configuration_name",
    "variable_name",
    "event_above_id",
    "event_above_peak_rank",
    "threshold",
]
metric_columns = ["peak_value"]
properties = {
    "description": "FIRO peak flow event rankings by location and quantile threshold",
    "group_by": ", ".join(group_by),
    "metrics": ", ".join(metric_columns),
}
for key, value in properties.items():
    ev.spark.sql(f"""
        ALTER TABLE iceberg.teehr.{table_name} SET TBLPROPERTIES ('{key}' = '{value}')
    """)

In [7]:
# Set event_heatmap table properties for API queryables
table_name = 'event_heatmap'
group_by = [
    "primary_location_id",
    "configuration_name",
    "variable_name",
    "event_id",
    "forecast_lead_time_bin",
    "threshold",
]
metric_columns = [
    "relative_bias",
    "root_mean_square_error",
    "pearson_correlation",
]
properties = {
    "description": "FIRO top events performance heatmap metrics by location and lead time bin",
    "group_by": ", ".join(group_by),
    "metrics": ", ".join(metric_columns),
}
for key, value in properties.items():
    ev.spark.sql(f"""
        ALTER TABLE iceberg.teehr.{table_name} SET TBLPROPERTIES ('{key}' = '{value}')
    """)

### Kill spark

In [8]:
spark.stop()